# `full-inference-engine` part 3 — memory after the warm-up fixes, and how close decode is to the memory roofline

Same T4, same fairness rules as parts 1 and 2 (FP16, identical token IDs, EOS ignored, matched KV
capacity, matched concurrency cap, NVML device-level memory).

Part 2 found that the repo engine held **7,291 MiB** resident for Qwen3-0.6B against vLLM's
**3,259 MiB**, and attributed almost all of the gap to CUDA-Graph warm-up rather than to the KV
pool: 255 graphs, a 2,223 MiB private graph pool, and 1,960 MiB of graph executables outside
PyTorch's allocator. The diagnosis was that warm-up enumerated rows and context buckets
independently and captured their whole product — including a 16-row × 16,384-token prefill shape
that needs **262,144 tokens of prefix against a 16,384-token pool**. That one unreachable shape
made the SDPA path gather ~1 GiB of K and V inside a single capture.

Three fixes landed since:

| fix | what it changes |
|---|---|
| reachable-shape guard | warm-up skips any `(rows, context)` whose `rows × context` exceeds the pool; the widest capture drops from 262,144 to 16,384 tokens |
| shared dummy pages | the padded CUDA-graph rows share pages and take distinct slots inside them, so the reservation is `ceil(rows / block_size)` pages instead of one page per row — invisible at 16-token pages, **10.9% of the pool** at the 256-token pages FlashAttention requires |
| pool-size refusal | an unaffordable KV pool raises at construction and names the `num_blocks` that fits, instead of dying inside the first capture |

**This notebook measures whether they worked, and what is left.**

### Part A — memory attribution, with the fix as an in-runtime A/B

Seven engine variants, one process each, NVML *and* PyTorch-allocator counters at every stage
(CUDA context → weights → KV pool → warm-up → `empty_cache`). Each variant also re-runs part 2's
speed shapes, so a memory win that costs latency cannot hide.

| variant | what changes |
|---|---|
| `asis` | current code, exactly as an application gets it |
| `unguarded` | `_shape_is_reachable` monkey-patched back to *always true* — **the pre-fix warm-up, in this same process, on this same card** |
| `capped` | warm-up context buckets capped at vLLM's `max_model_len` (one attribute, set before `warmup()`) |
| `no_fused` | `fused_step=False` |
| `no_prefill_graphs` | `prefill_cuda_graphs=False` |
| `no_graphs` | no CUDA graphs at all |
| `pages256` | 256-token pages at the same pool capacity — where the dummy-row reservation used to cost 10.9% of the KV pool |
| `pages256_unshared` | the same 256-token pages with the **pre-fix** reservation: one whole page per padded graph row |

`asis` vs `unguarded` and `pages256` vs `pages256_unshared` are the measurements that matter: one
predicate or one reservation rule changed, everything else identical. Part 2's published numbers are carried in the results cell as a cross-run check, not as
the comparison.

A vLLM row is measured in the same runtime at the same KV capacity, so the remaining gap is a
number rather than arithmetic between two notebooks.

### Part B — bandwidth: where decode actually sits against the roofline

For each `(batch, context)` operating point, the engine is driven into steady-state decode and a
window of decode-only steps is timed with one CUDA event pair. Bytes per step are the
checkpoint's own decode-read weights plus the KV the batch holds — counted both as the ideal
(each GQA group read once) and as implemented (the per-head kernel reads each group once per
query head). Divided by the step time that gives achieved GB/s, and divided again by a measured
fp16 GEMV probe it gives the fraction of this card's attainable rate.

Every window is validated before it is reported: all steps decode-only, batch never shrank, zero
CUDA-graph captures inside the window. A point that fails those checks says so instead of being
quietly averaged in.

**Runtime:** Part A ≈ 35–50 min (16 processes), the vLLM row ≈ 6 min including install, Part B
≈ 10–15 min. Trim `MODELS_A` / `MODELS_B` to one model to halve it.

In [ ]:
# ---------------- configuration ----------------
MODELS_A = ["Qwen/Qwen3-0.6B", "Qwen/Qwen3-1.7B"]   # memory attribution
VARIANTS = ["asis", "unguarded", "capped", "no_fused", "no_prefill_graphs", "no_graphs",
            "pages256", "pages256_unshared"]
MODELS_B = ["Qwen/Qwen3-0.6B", "Qwen/Qwen3-1.7B"]   # bandwidth / roofline sweep

# Part A: the same configuration parts 1 and 2 used, so the numbers are comparable.
KV_TOKENS = 16384        # KV pool capacity in tokens (1.75 GiB for a 28-layer, 8-KV-head Qwen3)
MAX_ACTIVE = 16          # concurrency cap, matched to vLLM's max_num_seqs
MAX_MODEL_LEN = 1024     # what vLLM was given; only the `capped` variant uses it
GEN_TOKENS = 128

# Part B: decode operating points. Context is per row, so 16 x 2048 needs 32k pool tokens.
BW_BATCH = [1, 4, 8, 16]
BW_CONTEXT = [128, 512, 2048]
BW_STEPS = 64            # decode steps per timed window
BW_WINDOWS = 3           # windows per point; the fastest valid one is reported
BW_PROBE_MB = 512        # bandwidth probe buffer size (must exceed L2 by a lot)

RUN_VLLM_REFERENCE = True     # same-runtime vLLM memory row; set False to skip the install
VLLM_SPEC = "vllm"            # pin e.g. "vllm==0.30.0" to match part 1
REPO_REF = "main"             # branch/tag/commit that carries the memory fixes
# If the fixes are not on GitHub yet, upload the working tree instead and point at it here
# (Colab: the folder icon on the left, then drag the zip in). The setup cell prefers this
# over cloning, so nothing has to be pushed to run this notebook.
REPO_ZIP = None               # e.g. "/content/full-inference-engine.zip"
TIMEOUT_S = 3000

WORKDIR = "/content/bench3"
REPO_DIR = "/content/full-inference-engine"
VLLM_PY = "/content/vllm-env/bin/python"
import os
for d in ("results_mem", "results_bw", "logs"):
    os.makedirs(f"{WORKDIR}/{d}", exist_ok=True)

## 1. Setup

The repo installs into Colab's own Python (it reuses Colab's torch/triton build deliberately);
vLLM gets an isolated `uv` venv. Skipped automatically if part 1 or 2 already prepared this runtime.

The cell **verifies that the post-fix code is present** and stops if it is not — running this
notebook against the pre-fix commit would just reproduce part 2.

If the fixes are not on GitHub yet, upload `full-inference-engine.zip` (drag it into Colab's file
pane) and set `REPO_ZIP = "/content/full-inference-engine.zip"` in the configuration cell. The setup
cell prefers the uploaded tree over cloning, so nothing has to be pushed to run this.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

if REPO_ZIP:
    # An uploaded working tree wins: this notebook has to be runnable before the fixes are
    # pushed anywhere. Accepts a zip made at the repo root or one with a single wrapper dir.
    import glob as _glob, shutil, zipfile
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    os.makedirs(REPO_DIR, exist_ok=True)
    with zipfile.ZipFile(REPO_ZIP) as archive:
        archive.extractall(REPO_DIR)
    if not os.path.isdir(f"{REPO_DIR}/engine"):
        inner = [d for d in _glob.glob(f"{REPO_DIR}/*") if os.path.isdir(f"{d}/engine")]
        assert inner, f"{REPO_ZIP} contains no engine/ package"
        for item in _glob.glob(f"{inner[0]}/*") + _glob.glob(f"{inner[0]}/.??*"):
            shutil.move(item, REPO_DIR)
    print("engine source from", REPO_ZIP)
else:
    !test -d {REPO_DIR} || git clone -q https://github.com/Vaibhav7711/full-inference-engine.git {REPO_DIR}
    !cd {REPO_DIR} && git fetch -q --all --tags && git checkout -q {REPO_REF} && (git pull -q --ff-only 2>/dev/null || true)
    !cd {REPO_DIR} && git log -1 --format='repo commit: %h %ad %s' --date=short
!python -c "import engine" 2>/dev/null || (pip install -q -r {REPO_DIR}/requirements-colab.txt nvidia-ml-py && pip install -q -e {REPO_DIR})
!pip install -q nvidia-ml-py uv

# --- refuse to run against the pre-fix engine -------------------------------------------
source = open(f"{REPO_DIR}/engine/batching/continuous_batching.py").read()
required = {
    "reachable-shape guard": "_shape_is_reachable",
    "shared dummy pages": "_graph_dummy_slots",
    "pool-size refusal": "Pass num_blocks <=",
}
missing = [name for name, token in required.items() if token not in source]
assert not missing, (
    f"this checkout is missing {missing}: it is the pre-fix engine, and part A would just "
    f"reproduce part 2. Push the memory-fix commits and set REPO_REF to a ref that has them.")
print("post-fix engine present:", ", ".join(required))

import torch
print("repo env torch", torch.__version__, "| cuda", torch.version.cuda)

In [ ]:
if RUN_VLLM_REFERENCE:
    !test -x {VLLM_PY} || (uv venv -q /content/vllm-env --python python3 && uv pip install -q --python {VLLM_PY} "{VLLM_SPEC}" nvidia-ml-py --torch-backend=auto)
    !{VLLM_PY} -c "import vllm, torch; print('vLLM', vllm.__version__, '| torch', torch.__version__)"
else:
    print("vLLM reference row skipped (RUN_VLLM_REFERENCE = False)")

## 2. Worker scripts

In [ ]:
%%writefile /content/bench3/bench3_common.py
"""Shared helpers for the part-3 workers (stdlib + pynvml only)."""
from __future__ import annotations

import statistics
import threading
import time


class GpuMemoryTracker:
    """Device-level GPU memory via NVML, polled in a background thread.

    NVML sees everything: CUDA context, weights, KV pool, CUDA-graph pools, CUDA-graph
    executables, allocator slack. PyTorch's own counters see only what its allocator
    manages, and the difference between the two is the whole point of part A.
    """

    def __init__(self, interval_s: float = 0.02):
        import pynvml

        self._nvml = pynvml
        pynvml.nvmlInit()
        self._handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        name = pynvml.nvmlDeviceGetName(self._handle)
        self.gpu_name = name.decode() if isinstance(name, bytes) else name
        self.total_mib = pynvml.nvmlDeviceGetMemoryInfo(self._handle).total / 2**20
        self.baseline_mib = self.used_mib()
        self.peak_mib = self.baseline_mib
        self._interval = interval_s
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._poll, daemon=True)

    def used_mib(self) -> float:
        return self._nvml.nvmlDeviceGetMemoryInfo(self._handle).used / 2**20

    def above_baseline(self) -> float:
        return self.used_mib() - self.baseline_mib

    def reset_peak(self) -> None:
        self.peak_mib = self.used_mib()

    def peak_above_baseline(self) -> float:
        return max(self.peak_mib, self.used_mib()) - self.baseline_mib

    def _poll(self) -> None:
        while not self._stop.is_set():
            self.peak_mib = max(self.peak_mib, self.used_mib())
            time.sleep(self._interval)

    def start(self) -> "GpuMemoryTracker":
        self._thread.start()
        return self

    def stop(self) -> None:
        self._stop.set()
        self._thread.join(timeout=2)
        self.peak_mib = max(self.peak_mib, self.used_mib())


def pct(values, q):
    if not values:
        return None
    ordered = sorted(values)
    index = min(len(ordered) - 1, max(0, round(q / 100 * (len(ordered) - 1))))
    return ordered[index]


def median(values):
    return statistics.median(values) if values else None

In [ ]:
%%writefile /content/bench3/mem_attrib3.py
"""Where does the repo engine's GPU memory go, after the warm-up fixes? One variant per process.

Stages are measured with NVML (device truth) and with PyTorch's allocator counters, so the
difference between them (`nvml - reserved`) is memory PyTorch does not manage: the CUDA
context, loaded kernel modules and CUDA-graph executables.

The `unguarded` variant monkey-patches `_shape_is_reachable` back to "always true", which is
exactly the pre-fix warm-up. It is the control for `asis`: same binary, same runtime, same
device, one predicate changed. That is a stronger comparison than reading the old notebook's
table, which was produced by a different commit on a different day.
"""
from __future__ import annotations

import argparse
import json
import time
import traceback

from bench3_common import GpuMemoryTracker

VARIANTS = {
    # current code, exactly as an application gets it
    "asis": dict(),
    # `_shape_is_reachable` forced true: the warm-up as it was before the fix
    "unguarded": dict(),
    # warm-up context buckets capped at the vLLM max_model_len (one attribute, set pre-warmup)
    "capped": dict(),
    # keep prefill graphs, drop the fused decode+prefill graphs
    "no_fused": dict(fused_step=False),
    # keep decode graphs only (fused graphs need prefill graphs, so they go too)
    "no_prefill_graphs": dict(prefill_cuda_graphs=False),
    # no CUDA graphs at all
    "no_graphs": dict(),
    # 256-token pages (what FlashAttention's paged API requires) - the shape in which the
    # per-row dummy-page reservation used to cost 10.9% of the pool
    "pages256": dict(),
    # the same 256-token pages with the pre-fix reservation: one whole page per padded row
    "pages256_unshared": dict(),
}


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--model", required=True)
    p.add_argument("--variant", required=True, choices=list(VARIANTS))
    p.add_argument("--kv-tokens", type=int, default=16384)
    p.add_argument("--max-active", type=int, default=16)
    p.add_argument("--max-model-len", type=int, default=1024)
    p.add_argument("--gen-tokens", type=int, default=128)
    p.add_argument("--runs", type=int, default=3)
    p.add_argument("--out", required=True)
    args = p.parse_args()
    tracker = GpuMemoryTracker().start()
    stages = {}
    try:
        import torch
        from engine.batching.continuous_batching import ContinuousBatchingEngine
        from engine.model import load_model
        from engine.runtime import GenerationRequest
        from engine.runtime.sampling import SamplingParams

        # Keep the real predicate before anything patches it: the skipped-shape census
        # below has to describe the shipped rule even in the `unguarded` arm.
        real_guard = ContinuousBatchingEngine._shape_is_reachable

        def snap(name):
            torch.cuda.synchronize()
            stages[name] = {
                "nvml_mib": round(tracker.above_baseline()),
                "allocated_mib": round(torch.cuda.memory_allocated() / 2**20),
                "reserved_mib": round(torch.cuda.memory_reserved() / 2**20),
            }

        torch.zeros(1, device="cuda")
        snap("1_cuda_context")
        loaded = load_model(args.model, dtype="float16")
        snap("2_weights")

        block = 256 if args.variant.startswith("pages256") else 16
        if args.variant == "pages256_unshared":
            # The pre-fix reservation: one whole page per padded CUDA-graph row, which is
            # 7 tokens of payload per 256-token page. Patched in before construction
            # because __init__ (and every reset) calls it.
            def unshared_reserve(self):
                self._graph_dummy_blocks = []
                self._graph_dummy_slots = []
                if not self.cuda_graph_batch_sizes:
                    return
                required = max(self.cuda_graph_batch_sizes) - 1
                if required <= 0:
                    return
                allocation = self.block_manager.reserve(
                    "__cuda_graph_dummy_rows__", required * self.block_size)
                if allocation is None:
                    raise ValueError("insufficient KV blocks to reserve CUDA-Graph dummy rows")
                self._graph_dummy_blocks = allocation.physical_block_ids
                self._graph_dummy_slots = [(page, 0) for page in self._graph_dummy_blocks]

            ContinuousBatchingEngine._reserve_graph_dummy_blocks = unshared_reserve
        kwargs = dict(VARIANTS[args.variant])
        if args.variant != "no_graphs":
            kwargs["cuda_graph_batch_sizes"] = tuple(
                b for b in (1, 2, 4, 8, 16, 32) if b <= args.max_active)
        engine = ContinuousBatchingEngine(
            loaded.model, loaded.tokenizer, loaded.device, max_active=args.max_active,
            num_blocks=args.kv_tokens // block, block_size=block, prefix_cache_blocks=0,
            **kwargs)
        snap("3_kv_pool_and_buffers")
        default_ctx_buckets = engine._warmup_context_buckets()
        if args.variant == "capped":
            # The only change: warm-up stops capturing graphs for contexts the workload
            # can never reach (vLLM gets the same max_model_len).
            engine.max_model_len = args.max_model_len
        if args.variant == "unguarded":
            ContinuousBatchingEngine._shape_is_reachable = staticmethod(
                lambda rows, context_len, pool_tokens: True)
        ctx_buckets = engine._warmup_context_buckets()
        pool_tokens = engine.key_pool[0].shape[0] * engine.block_size
        t0 = time.perf_counter()
        summary = engine.warmup()
        torch.cuda.synchronize()
        warmup_s = time.perf_counter() - t0
        snap("4_after_warmup")
        torch.cuda.empty_cache()
        snap("5_after_empty_cache")
        resident = tracker.above_baseline()
        ContinuousBatchingEngine._shape_is_reachable = staticmethod(real_guard)

        # ---- what warm-up actually captured, and what the guard refused ---------------
        kv_bytes_per_token = 2 * engine.num_kv_heads * engine.head_dim * 2   # K + V, fp16

        def gather_mib(rows, context_len):
            """K+V the SDPA prefill path gathers for one layer at this captured shape."""
            return rows * context_len * kv_bytes_per_token / 2**20

        def mask_mib(rows, context_len):
            return rows * engine.prefill_chunk_size * context_len / 2**20    # bool

        prefill_keys = [(rows, ctx) for rows, _attn, ctx in engine._prefill_graphs]
        fused_keys = [(prows, ctx) for _d, prows, _attn, ctx, _bn, _nw in engine._fused_graphs]
        captured = prefill_keys + fused_keys
        widest = max(captured, key=lambda rc: rc[0] * rc[1], default=(0, 0))
        enumerated = reachable = 0
        contexts = ctx_buckets if engine.prefill_attention == "sdpa" else [0]
        for rows in (1,) + tuple(engine.cuda_graph_batch_sizes):
            for context_len in contexts:
                enumerated += 1
                reachable += bool(real_guard(rows, context_len, pool_tokens))
        dummy_tokens = len(engine._graph_dummy_blocks) * engine.block_size
        geometry = {
            "block_size": engine.block_size,
            "pool_tokens": pool_tokens,
            "kv_bytes_per_token_all_layers": kv_bytes_per_token * engine.num_layers,
            "row_context_pairs_enumerated": enumerated,
            "row_context_pairs_reachable": reachable,
            "widest_captured_rows": widest[0],
            "widest_captured_context": widest[1],
            "widest_captured_tokens": widest[0] * widest[1],
            "largest_gather_mib_per_layer": round(gather_mib(*widest), 1),
            "largest_mask_mib_per_layer": round(mask_mib(*widest), 1),
            "dummy_row_pages": len(engine._graph_dummy_blocks),
            "dummy_row_tokens": dummy_tokens,
            "dummy_rows": len(engine._graph_dummy_slots),
            "dummy_pct_of_pool": round(100 * dummy_tokens / pool_tokens, 2),
        }

        # ---- speed check on the original benchmark's shapes --------------------------
        vocab = int(loaded.model.config.vocab_size)
        gen = torch.Generator().manual_seed(0)
        rand_ids = lambda n: torch.randint(1000, vocab - 1000, (n,), generator=gen).tolist()
        sp = SamplingParams(ignore_eos=True)

        def run(prompt_lists, n):
            engine.reset()
            reqs = [GenerationRequest(request_id=f"r{i}", prompt_token_count=len(ids),
                                      max_new_tokens=n, prompt_token_ids=list(ids), sampling=sp)
                    for i, ids in enumerate(prompt_lists)]
            torch.cuda.synchronize()
            t = time.perf_counter()
            for r in reqs:
                if engine.submit(r) is False:
                    raise RuntimeError("rejected")
            while engine.has_unfinished_requests:
                engine.step()
            torch.cuda.synchronize()
            return (time.perf_counter() - t) * 1000, reqs

        def median_of(fn):
            fn()
            return sorted(fn()[0] for _ in range(args.runs))[args.runs // 2]

        tracker.reset_peak()
        single = [rand_ids(512)]
        ttft = median_of(lambda: run(single, 1))
        e2e = median_of(lambda: run(single, args.gen_tokens))
        batch = [rand_ids(256) for _ in range(args.max_active)]
        batch_ms = median_of(lambda: run(batch, args.gen_tokens))
        speed = {
            "ttft_512_ms": ttft,
            "tpot_ms": (e2e - ttft) / (args.gen_tokens - 1),
            "batch_tok_s": args.max_active * args.gen_tokens / (batch_ms / 1000),
            "lazy_graph_captures_during_run": engine.lazy_graph_captures,
        }
        peak = tracker.peak_above_baseline()
        tracker.stop()
        w, k = stages["3_kv_pool_and_buffers"], stages["4_after_warmup"]
        result = {
            "ok": True, "model": args.model, "variant": args.variant, "gpu": tracker.gpu_name,
            "stages": stages, "warmup_s": round(warmup_s, 1), "warmup_summary": {
                key: summary.get(key) for key in ("graphs", "prefill_graphs", "fused_graphs",
                                                  "prefill_graph_unsupported")},
            "prefill_attention": engine.prefill_attention,
            "decode_attention": engine.decode_attention,
            "context_buckets_default": default_ctx_buckets,
            "context_buckets_used": ctx_buckets,
            "graph_geometry": geometry,
            "attribution_mib": {
                "cuda_context": stages["1_cuda_context"]["nvml_mib"],
                "weights": stages["2_weights"]["allocated_mib"] - stages["1_cuda_context"]["allocated_mib"],
                "kv_pool_and_buffers": w["allocated_mib"] - stages["2_weights"]["allocated_mib"],
                "graph_pool_and_allocator_cache": k["reserved_mib"] - w["reserved_mib"]
                                                  - (k["allocated_mib"] - w["allocated_mib"]),
                "warmup_outside_allocator": (k["nvml_mib"] - w["nvml_mib"])
                                            - (k["reserved_mib"] - w["reserved_mib"]),
            },
            "resident_mib": round(resident), "peak_mib": round(peak), "speed": speed,
        }
    except Exception as error:
        tracker.stop()
        result = {"ok": False, "model": args.model, "variant": args.variant,
                  "error": repr(error), "traceback": traceback.format_exc()[-5000:],
                  "stages": stages}
    with open(args.out, "w") as f:
        json.dump(result, f, indent=2)
    print("saved", args.out, flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/bench3/bandwidth_probe.py
"""How close is a decode step to this device's measured memory bandwidth?

Two halves, both measured on the card in front of us:

1. **The rate.** `benchmarks.kernels.roofline.measure_bandwidth` runs three probes on
   buffers far larger than L2. The fp16 matrix-vector product is the representative one,
   because that *is* the decode operation: read an enormous matrix, touch a tiny vector,
   write almost nothing.
2. **The traffic.** For each (batch, context) operating point the engine is driven into
   steady-state decode and a window of `--steps` decode-only steps is timed with one CUDA
   event pair. Bytes per step are the checkpoint's own decode-read weights plus the KV the
   batch's contexts hold - counted twice over, once as the ideal (each GQA group read once)
   and once as implemented (the per-head kernel reads each group once per query head).

`achieved GB/s` divided by the gemv probe is then the fraction of the attainable rate this
engine reaches, and `multiple of floor` is the same statement as a latency.

The window is validated, not assumed: a point is only reported if every timed step was
decode-only (no fused prefill rows), the batch never shrank, and no CUDA graph was captured
inside the window.
"""
from __future__ import annotations

import argparse
import json
import time
import traceback

from bench3_common import GpuMemoryTracker


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--model", required=True)
    p.add_argument("--batch", type=int, nargs="+", default=[1, 4, 8, 16])
    p.add_argument("--context", type=int, nargs="+", default=[128, 512, 2048])
    p.add_argument("--steps", type=int, default=64)
    p.add_argument("--windows", type=int, default=3)
    p.add_argument("--probe-mb", type=int, default=512)
    p.add_argument("--block-size", type=int, default=16)
    p.add_argument("--decode-attention", default="per_head")
    p.add_argument("--out", required=True)
    args = p.parse_args()
    tracker = GpuMemoryTracker().start()
    try:
        import torch
        from benchmarks.kernels.roofline import measure_bandwidth, measure_weight_bytes
        from engine.batching.continuous_batching import ContinuousBatchingEngine
        from engine.kernels.device import current_device
        from engine.model import load_model
        from engine.runtime import GenerationRequest
        from engine.runtime.sampling import SamplingParams

        profile = current_device()
        probes = [{"name": pr.name, "description": pr.description, "gb_per_s": pr.gb_per_s,
                   "ms": pr.ms, "bytes_moved": pr.bytes_moved}
                  for pr in measure_bandwidth(args.probe_mb)]
        gemv = next(pr for pr in probes if pr["name"] == "gemv")
        rate = gemv["gb_per_s"]
        print(f"[bw] {tracker.gpu_name}: gemv probe {rate:.0f} GB/s", flush=True)

        loaded = load_model(args.model, dtype="float16")
        weights = measure_weight_bytes(loaded.model)
        weight_read = weights.read_per_decode_step

        batches = sorted(set(args.batch))
        max_batch, max_context = max(batches), max(args.context)
        # Every row needs its prompt plus the whole timed window resident, and the pool
        # also carries the dummy rows the graph buckets reserve.
        needed = max_batch * (max_context + args.steps * args.windows + 64)
        num_blocks = -(-int(needed * 1.15) // args.block_size)
        engine = ContinuousBatchingEngine(
            loaded.model, loaded.tokenizer, loaded.device, max_active=max_batch,
            num_blocks=num_blocks, block_size=args.block_size, prefix_cache_blocks=0,
            decode_attention=args.decode_attention,
            cuda_graph_batch_sizes=tuple(batches))
        # Warm-up captures graphs for every context bucket the pool can reach; this probe
        # only ever reaches `max_context`, and capturing past it would cost minutes.
        engine.max_model_len = min(int(engine.max_model_len or max_context * 2),
                                   max(512, max_context * 2))
        t0 = time.perf_counter()
        warm = engine.warmup()
        torch.cuda.synchronize()
        warmup_s = time.perf_counter() - t0
        print(f"[bw] warm-up {warmup_s:.0f}s: {warm['graphs']}/{warm['prefill_graphs']}"
              f"/{warm['fused_graphs']} decode/prefill/fused graphs", flush=True)

        vocab = int(loaded.model.config.vocab_size)
        generator = torch.Generator().manual_seed(0)
        greedy_ignore_eos = SamplingParams(ignore_eos=True)
        cfg = loaded.model.config
        kv_heads = int(getattr(cfg, "num_key_value_heads", cfg.num_attention_heads))
        q_heads = int(cfg.num_attention_heads)
        head_dim = int(getattr(cfg, "head_dim", cfg.hidden_size // cfg.num_attention_heads))
        layers = int(cfg.num_hidden_layers)
        kv_bytes_per_context_token = 2 * kv_heads * head_dim * 2 * layers   # K+V, fp16, all layers
        replication = q_heads / kv_heads

        def drive_to_steady_state(batch, context):
            engine.reset()
            for index in range(batch):
                ids = torch.randint(1000, vocab - 1000, (context,), generator=generator).tolist()
                ok = engine.submit(GenerationRequest(
                    request_id=f"bw{index}", prompt_token_count=context,
                    max_new_tokens=args.steps * args.windows + 32,
                    prompt_token_ids=ids, sampling=greedy_ignore_eos))
                if ok is False:
                    raise RuntimeError(f"submit rejected at batch {batch}, context {context}")
            guard = 0
            while guard < 40_000:
                engine.step()
                guard += 1
                stats = engine.stats_snapshot()
                if stats["decode_batch"] == batch and engine.last_step_prefill_tokens == 0:
                    return stats
            raise RuntimeError("never reached steady-state decode")

        def time_window(steps):
            """One event-timed run of `steps` decode steps; returns ms and what it did."""
            before = engine.stats_snapshot()
            lazy_before = engine.lazy_graph_captures
            fused_before, decode_before = engine.fused_steps, engine.decode_only_steps
            torch.cuda.synchronize()
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            for _ in range(steps):
                engine.step()
            end.record()
            torch.cuda.synchronize()
            after = engine.stats_snapshot()
            return {
                "ms": start.elapsed_time(end),
                "steps": steps,
                "decode_only_steps": engine.decode_only_steps - decode_before,
                "fused_steps": engine.fused_steps - fused_before,
                "lazy_captures": engine.lazy_graph_captures - lazy_before,
                "batch_before": before["decode_batch"], "batch_after": after["decode_batch"],
                "context_tokens_before": before["decode_context_tokens"],
                "context_tokens_after": after["decode_context_tokens"],
            }

        points = []
        for batch in batches:
            for context in args.context:
                if batch * (context + args.steps * args.windows + 64) > num_blocks * args.block_size:
                    continue
                try:
                    drive_to_steady_state(batch, context)
                    time_window(8)                       # settle: fill caches, no timing kept
                    windows = [time_window(args.steps) for _ in range(args.windows)]
                except Exception as error:                # keep the rest of the sweep
                    points.append({"batch": batch, "context": context, "ok": False,
                                   "error": repr(error)})
                    print(f"[bw] batch {batch} ctx {context}: FAILED {error!r}", flush=True)
                    continue
                clean = [w for w in windows
                         if w["fused_steps"] == 0 and w["lazy_captures"] == 0
                         and w["decode_only_steps"] == w["steps"]
                         and w["batch_before"] == w["batch_after"] == batch]
                chosen = min(clean or windows, key=lambda w: w["ms"] / w["steps"])
                step_ms = chosen["ms"] / chosen["steps"]
                mean_context = (chosen["context_tokens_before"]
                                + chosen["context_tokens_after"]) / 2
                kv_ideal = kv_bytes_per_context_token * mean_context
                kv_implemented = kv_ideal * replication
                seconds = step_ms / 1000
                point = {
                    "batch": batch, "context": context, "ok": True,
                    "windows_clean": len(clean), "windows": len(windows),
                    "step_ms": step_ms,
                    "step_ms_all": [w["ms"] / w["steps"] for w in windows],
                    "mean_context_tokens_total": mean_context,
                    "mean_context_per_row": mean_context / batch,
                    "weight_bytes": weight_read,
                    "kv_bytes_ideal": kv_ideal,
                    "kv_bytes_implemented": kv_implemented,
                    "bytes_ideal": weight_read + kv_ideal,
                    "bytes_implemented": weight_read + kv_implemented,
                    "achieved_gb_per_s_ideal": (weight_read + kv_ideal) / seconds / 1e9,
                    "achieved_gb_per_s_implemented": (weight_read + kv_implemented) / seconds / 1e9,
                    "floor_ms_ideal": (weight_read + kv_ideal) / (rate * 1e9) * 1000,
                    "floor_ms_implemented": (weight_read + kv_implemented) / (rate * 1e9) * 1000,
                    "lazy_captures": chosen["lazy_captures"],
                    "fused_steps_in_window": chosen["fused_steps"],
                }
                point["pct_of_probe_ideal"] = 100 * point["achieved_gb_per_s_ideal"] / rate
                point["pct_of_probe_implemented"] = (
                    100 * point["achieved_gb_per_s_implemented"] / rate)
                point["multiple_of_floor_ideal"] = step_ms / point["floor_ms_ideal"]
                point["multiple_of_floor_implemented"] = step_ms / point["floor_ms_implemented"]
                point["tokens_per_s_aggregate"] = batch / seconds
                points.append(point)
                print(f"[bw] batch {batch:2d} ctx {context:5d}: {step_ms:7.3f} ms/step  "
                      f"{point['achieved_gb_per_s_implemented']:6.1f} GB/s  "
                      f"{point['pct_of_probe_implemented']:5.1f}% of probe  "
                      f"{point['multiple_of_floor_implemented']:.2f}x floor "
                      f"({len(clean)}/{len(windows)} clean)", flush=True)

        peak = tracker.peak_above_baseline()
        resident = tracker.above_baseline()
        tracker.stop()
        result = {
            "ok": True, "model": args.model, "gpu": tracker.gpu_name,
            "device_profile": profile.to_dict() if profile else None,
            "probes": probes, "achieved_gb_per_s": rate,
            "weight_bytes": {"layers": weights.layers, "lm_head": weights.lm_head,
                             "other": weights.other, "embedding_table": weights.embedding_table,
                             "lm_head_is_tied": weights.lm_head_is_tied,
                             "read_per_decode_step": weight_read},
            "geometry": {"layers": layers, "q_heads": q_heads, "kv_heads": kv_heads,
                         "head_dim": head_dim, "replication": replication,
                         "kv_bytes_per_context_token": kv_bytes_per_context_token},
            "engine": {"block_size": args.block_size, "num_blocks": num_blocks,
                       "max_active": max_batch, "decode_attention": engine.decode_attention,
                       "prefill_attention": engine.prefill_attention,
                       "decode_backend": getattr(engine.decode_backend, "name", None),
                       "decode_cuda_graphs": engine.decode_cuda_graphs,
                       "warmup_s": round(warmup_s, 1), "warmup_summary": {
                           k: warm.get(k) for k in ("graphs", "prefill_graphs", "fused_graphs")},
                       "max_model_len_used": engine.max_model_len},
            "steps_per_window": args.steps, "windows": args.windows,
            "points": points,
            "resident_mib": round(resident), "peak_mib": round(peak),
        }
    except Exception as error:
        tracker.stop()
        result = {"ok": False, "model": args.model, "error": repr(error),
                  "traceback": traceback.format_exc()[-5000:]}
    with open(args.out, "w") as f:
        json.dump(result, f, indent=2)
    print("saved", args.out, flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/bench3/vllm_mem_ref.py
"""vLLM's resident memory at the same KV capacity, measured in the same runtime.

Not a throughput comparison - part 1 already did that, and this is a 1024-token
`max_model_len` micro-configuration rather than a production deployment. The only job here
is a same-runtime reference point for the memory column, so "the gap closed to X MiB" is a
measurement rather than an arithmetic between two notebooks.
"""
from __future__ import annotations

import argparse
import json
import time
import traceback

from bench3_common import GpuMemoryTracker


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--model", required=True)
    p.add_argument("--kv-bytes", type=int, required=True)
    p.add_argument("--max-model-len", type=int, default=1024)
    p.add_argument("--max-active", type=int, default=16)
    p.add_argument("--gen-tokens", type=int, default=128)
    p.add_argument("--runs", type=int, default=3)
    p.add_argument("--gpu-memory-utilization", type=float, default=0.90)
    p.add_argument("--out", required=True)
    args = p.parse_args()
    tracker = GpuMemoryTracker().start()
    try:
        import vllm
        from transformers import AutoConfig
        from vllm import LLM, SamplingParams

        t0 = time.perf_counter()
        kwargs = dict(model=args.model, dtype="half", max_model_len=args.max_model_len,
                      max_num_seqs=args.max_active, enable_prefix_caching=False,
                      gpu_memory_utilization=args.gpu_memory_utilization,
                      kv_cache_memory_bytes=args.kv_bytes, seed=0, disable_log_stats=True)
        kv_matched = True
        try:
            llm = LLM(**kwargs)
        except TypeError:
            kwargs.pop("kv_cache_memory_bytes")
            kv_matched = False
            llm = LLM(**kwargs)
        startup_s = time.perf_counter() - t0
        after_load = tracker.above_baseline()

        cfg = AutoConfig.from_pretrained(args.model)
        vocab = int(cfg.vocab_size)
        import random
        rng = random.Random(0)
        rand_ids = lambda n: [rng.randrange(1000, vocab - 1000) for _ in range(n)]

        def run(prompt_lists, n):
            params = SamplingParams(temperature=0.0, max_tokens=n, ignore_eos=True)
            prompts = [{"prompt_token_ids": list(ids)} for ids in prompt_lists]
            start = time.perf_counter()
            llm.generate(prompts, params, use_tqdm=False)
            return (time.perf_counter() - start) * 1000

        def median_of(fn):
            fn()
            return sorted(fn() for _ in range(args.runs))[args.runs // 2]

        tracker.reset_peak()
        single = [rand_ids(512)]
        ttft = median_of(lambda: run(single, 1))
        e2e = median_of(lambda: run(single, args.gen_tokens))
        batch = [rand_ids(256) for _ in range(args.max_active)]
        batch_ms = median_of(lambda: run(batch, args.gen_tokens))
        resident = tracker.above_baseline()
        peak = tracker.peak_above_baseline()
        tracker.stop()
        result = {
            "ok": True, "model": args.model, "variant": "vllm_reference",
            "gpu": tracker.gpu_name, "vllm_version": vllm.__version__,
            "kv_capacity_matched": kv_matched, "kv_bytes": args.kv_bytes,
            "max_model_len": args.max_model_len, "startup_s": round(startup_s, 1),
            "resident_after_load_mib": round(after_load),
            "resident_mib": round(resident), "peak_mib": round(peak),
            "speed": {"ttft_512_ms": ttft,
                      "tpot_ms": (e2e - ttft) / (args.gen_tokens - 1),
                      "batch_tok_s": args.max_active * args.gen_tokens / (batch_ms / 1000)},
        }
    except Exception as error:
        tracker.stop()
        result = {"ok": False, "model": args.model, "variant": "vllm_reference",
                  "error": repr(error), "traceback": traceback.format_exc()[-5000:]}
    with open(args.out, "w") as f:
        json.dump(result, f, indent=2)
    print("saved", args.out, flush=True)


if __name__ == "__main__":
    main()

In [ ]:
import json, subprocess, sys, time

def gpu_used_mib():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout
    return float(out.strip().splitlines()[0])

def wait_for_idle_gpu(limit_mib=800, timeout_s=90):
    """Never start a memory measurement while the previous process is still releasing."""
    start = time.time()
    while gpu_used_mib() > limit_mib and time.time() - start < timeout_s:
        time.sleep(2)

def run_proc(cmd, log, out, env_repo, tags=("[bw]", "[repo]", "saved")):
    env = dict(os.environ, PYTHONPATH=WORKDIR + (f":{REPO_DIR}" if env_repo else ""),
               TOKENIZERS_PARALLELISM="false")
    wait_for_idle_gpu()
    if os.path.exists(out):
        os.remove(out)
    t0 = time.time()
    with open(log, "w") as logf:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                                env=env, cwd=REPO_DIR if env_repo else WORKDIR)
        try:
            for line in proc.stdout:
                logf.write(line)
                if line.startswith(tags) or "Error" in line:
                    print("  " + line.rstrip(), flush=True)
            proc.wait(timeout=TIMEOUT_S)
        except subprocess.TimeoutExpired:
            proc.kill(); print("  timed out")
    if not os.path.exists(out):
        json.dump({"ok": False, "error": f"exit code {proc.returncode}; see {log}"}, open(out, "w"))
    r = json.load(open(out))
    print(f"  done in {time.time()-t0:.0f}s: "
          f"{'ok' if r.get('ok') else 'FAILED: ' + str(r.get('error'))[:200]}")
    return r

## Part A — where the memory goes now

Both engines still preallocate the same 16,384-token KV pool (1.75 GiB for Qwen3-0.6B), so the KV
pool was never the gap and is not expected to move. What should move is everything warm-up spends:
the private CUDA-graph pool, the graph executables outside PyTorch's allocator, and the time.

What to look for:

- `graph shapes` — `widest captured` should now be at most the pool's token count. `unguarded`
  should show 262,144 for a 16,384-token pool, and `asis` should show 16,384.
- `largest gather MiB` — the K+V one capture materialises for a single layer. 1,024 MiB unguarded,
  64 MiB guarded.
- `pairs kept` — how many of warm-up's enumerated `(rows, context)` pairs survive the guard.
- `dummy tok (% pool)` — tokens reserved for padded graph rows, and what fraction of the pool they
  cost. At 16-token pages the fix is invisible (112 → 16 tokens). At 256-token pages it is the
  difference between `pages256_unshared` (15 pages = 3,840 tokens, 23% of the pool) and `pages256`
  (1 page = 256 tokens, 1.6%) — KV capacity handed back to the workload.
- `TTFT / TPOT / B16 tok/s` — must not regress against `unguarded`. A memory fix that costs speed
  is a trade, not a fix.

In [ ]:
for model in MODELS_A:
    for variant in VARIANTS:
        short = model.split("/")[-1]
        out = f"{WORKDIR}/results_mem/{short}_{variant}.json"
        print(f"\n▶ memory probe {short} / {variant}")
        run_proc([sys.executable, f"{WORKDIR}/mem_attrib3.py", "--model", model,
                  "--variant", variant, "--kv-tokens", str(KV_TOKENS),
                  "--max-active", str(MAX_ACTIVE), "--max-model-len", str(MAX_MODEL_LEN),
                  "--gen-tokens", str(GEN_TOKENS), "--out", out],
                 f"{WORKDIR}/logs/mem_{short}_{variant}.log", out, env_repo=True)

In [ ]:
from transformers import AutoConfig

def kv_bytes_for(model, tokens):
    """Bytes vLLM must hold to match the repo's KV capacity, from the model's own geometry."""
    cfg = AutoConfig.from_pretrained(model)
    kv_heads = getattr(cfg, "num_key_value_heads", cfg.num_attention_heads)
    head_dim = getattr(cfg, "head_dim", cfg.hidden_size // cfg.num_attention_heads)
    return 2 * cfg.num_hidden_layers * kv_heads * head_dim * 2 * tokens

if RUN_VLLM_REFERENCE:
    for model in MODELS_A:
        short = model.split("/")[-1]
        out = f"{WORKDIR}/results_mem/{short}_vllm_reference.json"
        kvb = kv_bytes_for(model, KV_TOKENS)
        print(f"\n▶ vLLM reference {short} (KV {kvb/2**20:.0f} MiB = {KV_TOKENS} tokens)")
        run_proc([VLLM_PY, f"{WORKDIR}/vllm_mem_ref.py", "--model", model,
                  "--kv-bytes", str(kvb), "--max-model-len", str(MAX_MODEL_LEN),
                  "--max-active", str(MAX_ACTIVE), "--gen-tokens", str(GEN_TOKENS), "--out", out],
                 f"{WORKDIR}/logs/vllm_{short}.log", out, env_repo=False)

In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt

# Part 2's published table (pre-fix commit, 25 Sep), carried only as a cross-run sanity check.
PART2 = {
    ("Qwen3-0.6B", "asis"):              dict(resident=7291, pool=2223, outside=1960, graphs="10/35/210", warm=43.9, ttft=58.63, tpot=7.87, tok_s=1079.05),
    ("Qwen3-0.6B", "capped"):            dict(resident=4313, pool=323,  outside=882,  graphs="10/15/90",  warm=16.8, ttft=59.47, tpot=7.90, tok_s=1051.23),
    ("Qwen3-0.6B", "no_fused"):          dict(resident=5619, pool=2221, outside=290,  graphs="10/35/0",   warm=11.9, ttft=61.54, tpot=8.04, tok_s=998.88),
    ("Qwen3-0.6B", "no_prefill_graphs"): dict(resident=3327, pool=147,  outside=72,   graphs="10/0/0",    warm=6.3,  ttft=162.58, tpot=7.96, tok_s=721.51),
    ("Qwen3-0.6B", "no_graphs"):         dict(resident=3105, pool=16,   outside=12,   graphs="0/0/0",     warm=2.0,  ttft=212.92, tpot=34.12, tok_s=339.33),
    ("Qwen3-1.7B", "asis"):              dict(resident=9497, pool=2278, outside=1960, graphs="10/35/210", warm=50.2, ttft=128.65, tpot=17.25, tok_s=523.05),
    ("Qwen3-1.7B", "capped"):            dict(resident=6497, pool=358,  outside=880,  graphs="10/15/90",  warm=20.4, ttft=132.83, tpot=19.69, tok_s=506.65),
    ("Qwen3-1.7B", "no_fused"):          dict(resident=7827, pool=2278, outside=290,  graphs="10/35/0",   warm=17.5, ttft=131.51, tpot=18.45, tok_s=485.84),
    ("Qwen3-1.7B", "no_prefill_graphs"): dict(resident=5461, pool=130,  outside=72,   graphs="10/0/0",    warm=7.9,  ttft=166.09, tpot=17.58, tok_s=470.71),
    ("Qwen3-1.7B", "no_graphs"):         dict(resident=5237, pool=-4,   outside=12,   graphs="0/0/0",     warm=3.4,  ttft=189.18, tpot=37.43, tok_s=305.48),
}

rows, vllm_rows, fails = [], [], []
for path in sorted(glob.glob(f"{WORKDIR}/results_mem/*.json")):
    r = json.load(open(path))
    if not r.get("ok"):
        fails.append(r); continue
    if r["variant"] == "vllm_reference":
        vllm_rows.append({
            "model": r["model"].split("/")[-1], "variant": "vLLM (reference)",
            "resident MiB": r["resident_mib"], "peak MiB": r["peak_mib"],
            "after load MiB": r["resident_after_load_mib"], "startup s": r["startup_s"],
            "KV matched": r["kv_capacity_matched"], "vLLM": r["vllm_version"],
            "TTFT@512 ms": r["speed"]["ttft_512_ms"], "TPOT ms": r["speed"]["tpot_ms"],
            "B16 tok/s": r["speed"]["batch_tok_s"]})
        continue
    a, s, g, q = r["attribution_mib"], r["speed"], r["warmup_summary"], r["graph_geometry"]
    model = r["model"].split("/")[-1]
    rows.append({
        "model": model, "variant": r["variant"],
        "resident MiB": r["resident_mib"], "peak MiB": r["peak_mib"],
        "context": a["cuda_context"], "weights": a["weights"], "KV+buffers": a["kv_pool_and_buffers"],
        "graph pool": a["graph_pool_and_allocator_cache"],
        "outside allocator": a["warmup_outside_allocator"],
        "graphs d/p/f": f"{g['graphs']}/{g['prefill_graphs']}/{g['fused_graphs']}",
        "pairs kept": f"{q['row_context_pairs_reachable']}/{q['row_context_pairs_enumerated']}",
        "widest rows×ctx": f"{q['widest_captured_rows']}×{q['widest_captured_context']}",
        "widest tokens": q["widest_captured_tokens"],
        "largest gather MiB": q["largest_gather_mib_per_layer"],
        "dummy tok": q["dummy_row_tokens"], "dummy % pool": q["dummy_pct_of_pool"],
        "pool tokens": q["pool_tokens"],
        "page": q["block_size"], "ctx buckets": len(r["context_buckets_used"]),
        "warm-up s": r["warmup_s"],
        "TTFT@512 ms": s["ttft_512_ms"], "TPOT ms": s["tpot_ms"], "B16 tok/s": s["batch_tok_s"],
        "lazy captures": s["lazy_graph_captures_during_run"],
    })
for r in fails:
    print("FAILED", r.get("model"), r.get("variant"), r.get("error"),
          str(r.get("traceback", ""))[-900:])

order = {v: i for i, v in enumerate(VARIANTS)}
mem = pd.DataFrame(rows).sort_values(
    ["model", "variant"], key=lambda c: c.map(order) if c.name == "variant" else c)
print("=== part A: memory attribution (MiB above an idle GPU) ===")
display(mem.round(2))
mem.to_csv(f"{WORKDIR}/memory_attribution_part3.csv", index=False)
if vllm_rows:
    print("=== vLLM reference, same runtime, same KV capacity ===")
    display(pd.DataFrame(vllm_rows).round(2))

# --- the measurement that matters: one predicate changed ---------------------------------
print("\n=== the fix, as an in-runtime A/B (unguarded = pre-fix warm-up) ===")
deltas = []
for model, df in mem.groupby("model"):
    by = df.set_index("variant")
    if not {"asis", "unguarded"} <= set(by.index):
        continue
    u, a = by.loc["unguarded"], by.loc["asis"]
    row = {"model": model}
    for label, key in [("resident MiB", "resident MiB"), ("graph pool", "graph pool"),
                       ("outside allocator", "outside allocator"),
                       ("largest gather MiB", "largest gather MiB"), ("warm-up s", "warm-up s"),
                       ("TTFT@512 ms", "TTFT@512 ms"), ("TPOT ms", "TPOT ms"),
                       ("B16 tok/s", "B16 tok/s")]:
        row[f"{label}: unguarded"] = u[key]
        row[f"{label}: asis"] = a[key]
        row[f"{label}: change %"] = round(100 * (a[key] - u[key]) / u[key], 1) if u[key] else None
    row["graphs unguarded"] = u["graphs d/p/f"]
    row["graphs asis"] = a["graphs d/p/f"]
    if vllm_rows:
        ref = next((v for v in vllm_rows if v["model"] == model), None)
        if ref:
            row["vLLM resident MiB"] = ref["resident MiB"]
            row["gap before MiB"] = u["resident MiB"] - ref["resident MiB"]
            row["gap after MiB"] = a["resident MiB"] - ref["resident MiB"]
    deltas.append(row)
if deltas:
    display(pd.DataFrame(deltas).set_index("model").T)
else:
    print("need both `asis` and `unguarded` to run this comparison")

# --- the second fix: one page per padded row, or one page for all of them ----------------
print("\n=== the dummy-page fix at 256-token pages (pages256_unshared = the pre-fix rule) ===")
pairs = []
for model, df in mem.groupby("model"):
    by = df.set_index("variant")
    if not {"pages256", "pages256_unshared"} <= set(by.index):
        continue
    old, new = by.loc["pages256_unshared"], by.loc["pages256"]
    pairs.append({
        "model": model, "page size": new["page"], "pool tokens": new["pool tokens"],
        "reserved tokens: before": old["dummy tok"], "reserved tokens: after": new["dummy tok"],
        "% of pool: before": old["dummy % pool"], "% of pool: after": new["dummy % pool"],
        "usable KV tokens: before": new["pool tokens"] - old["dummy tok"],
        "usable KV tokens: after": new["pool tokens"] - new["dummy tok"],
        "resident MiB: before": old["resident MiB"], "resident MiB: after": new["resident MiB"],
        "B16 tok/s: before": round(old["B16 tok/s"], 1),
        "B16 tok/s: after": round(new["B16 tok/s"], 1)})
if pairs:
    display(pd.DataFrame(pairs).set_index("model").T)
else:
    print("need both `pages256` and `pages256_unshared` to run this comparison")

# --- cross-run check against part 2 ------------------------------------------------------
print("\n=== cross-run check: this run vs part 2's published table (different commit/day) ===")
checks = []
for _, r in mem.iterrows():
    old = PART2.get((r["model"], r["variant"]))
    if not old:
        continue
    checks.append({"model": r["model"], "variant": r["variant"],
                   "resident part2": old["resident"], "resident now": r["resident MiB"],
                   "Δ MiB": r["resident MiB"] - old["resident"],
                   "graphs part2": old["graphs"], "graphs now": r["graphs d/p/f"],
                   "graph pool part2": old["pool"], "graph pool now": r["graph pool"],
                   "warm-up part2 s": old["warm"], "warm-up now s": r["warm-up s"]})
display(pd.DataFrame(checks))

parts = ["context", "weights", "KV+buffers", "graph pool", "outside allocator"]
if len(mem):
    fig, axes = plt.subplots(1, mem.model.nunique(), figsize=(8 * mem.model.nunique(), 4.8),
                             squeeze=False)
    for ax, (model, df) in zip(axes[0], mem.groupby("model")):
        df = df.sort_values("variant", key=lambda c: c.map(order))
        bottom = [0] * len(df)
        for part in parts:
            vals = df[part].clip(lower=0).tolist()
            ax.bar(df.variant, vals, bottom=bottom, label=part)
            bottom = [b + v for b, v in zip(bottom, vals)]
        ax.plot(df.variant, df["resident MiB"], "k_", markersize=28, label="NVML resident")
        ref = next((v["resident MiB"] for v in vllm_rows if v["model"] == model), None)
        if ref:
            ax.axhline(ref, color="crimson", ls="--", lw=1, label=f"vLLM {ref} MiB")
        ax.set_title(f"{model}: engine memory by component"); ax.set_ylabel("MiB")
        ax.tick_params(axis="x", rotation=25)
    axes[0][0].legend(fontsize=8)
    fig.tight_layout(); fig.savefig(f"{WORKDIR}/memory_attribution_part3.png", dpi=130); plt.show()

**How to read it.**

`graph pool` is the growth of PyTorch's *reserved* memory during warm-up beyond what live tensors
need — the CUDA-graph private pool, which `empty_cache()` cannot release while the graphs exist.
`outside allocator` is NVML growth during warm-up that PyTorch's allocator never sees: mostly
instantiated graph executables and loaded kernel modules. Divide it by the graph count for a
per-graph cost; it should be roughly constant per graph, which is why cutting graph *count* and
cutting graph *pool bytes* are two different levers.

`lazy captures` must be **0** everywhere. A non-zero value means that variant had to capture a
graph mid-run — a ~150 ms latency spike inside a timed window, and the reason part 2's first fused
measurement reported a p99 regression that turned out to be warm-up coverage.

The cross-run check against part 2 is there to catch a changed runtime (different driver, different
Colab torch build), not to be the result. If `unguarded` here does not land near part 2's `asis`,
trust the in-runtime A/B and distrust the cross-run delta.

## Part B — bandwidth: the decode floor, measured on this card

A decode step is memory-bound at these batch sizes: every weight is read once to produce one token
per sequence, and the arithmetic is a rounding error beside that traffic. So the floor is
*(bytes that must move) / (bytes per second this card actually delivers)* — and both halves are
measured here rather than quoted from a datasheet.

The GEMV probe is the representative rate because it *is* the decode pattern: read an enormous
matrix, touch a tiny vector, write almost nothing. Buffers are far larger than the T4's 4 MB L2,
so nothing is served from cache.

Two byte counts are reported for every point:

- **ideal** — each GQA group's K/V read once. This is the floor a perfect kernel would hit.
- **as implemented** — the per-head decode kernel runs one program per *query* head, so a group is
  read once per query head sharing it (×2 on Qwen3: 16 query heads, 8 KV heads). Part 2 measured a
  group-sharing variant at 0.95–1.06×, i.e. nothing, because L2 was already serving the duplicate
  read — so the honest achieved-bandwidth number is the ideal one and the *DRAM* traffic is closer
  to ideal than the as-implemented count suggests. Both are printed; neither is hidden.

In [ ]:
for model in MODELS_B:
    short = model.split("/")[-1]
    out = f"{WORKDIR}/results_bw/{short}.json"
    print(f"\n▶ bandwidth / roofline {short}")
    run_proc([sys.executable, f"{WORKDIR}/bandwidth_probe.py", "--model", model,
              "--batch", *map(str, BW_BATCH), "--context", *map(str, BW_CONTEXT),
              "--steps", str(BW_STEPS), "--windows", str(BW_WINDOWS),
              "--probe-mb", str(BW_PROBE_MB), "--out", out],
             f"{WORKDIR}/logs/bw_{short}.log", out, env_repo=True)

In [ ]:
bw = {}
for path in sorted(glob.glob(f"{WORKDIR}/results_bw/*.json")):
    r = json.load(open(path))
    if not r.get("ok"):
        print("FAILED", r.get("model"), r.get("error"), str(r.get("traceback", ""))[-900:])
        continue
    bw[r["model"].split("/")[-1]] = r

for model, r in bw.items():
    print(f"\n=== {model} on {r['gpu']} ===")
    probes = pd.DataFrame([{"probe": p["name"], "GB/s": round(p["gb_per_s"], 1),
                            "what it does": p["description"]} for p in r["probes"]])
    display(probes)
    w = r["weight_bytes"]
    print(f"decode-read weights: {w['read_per_decode_step']/1e6:.1f} MB "
          f"(layers {w['layers']/1e6:.1f} + lm_head {w['lm_head']/1e6:.1f}"
          f"{' tied' if w['lm_head_is_tied'] else ''} + other {w['other']/1e6:.1f}); "
          f"embedding table {w['embedding_table']/1e6:.1f} MB is gathered, not swept")
    g, e = r["geometry"], r["engine"]
    print(f"KV: {g['kv_bytes_per_context_token']/1024:.0f} KiB per context token across "
          f"{g['layers']} layers ({g['kv_heads']} KV heads x {g['head_dim']}, fp16), "
          f"query-head replication x{g['replication']:.0f}")
    print(f"engine: {e['decode_attention']} decode / {e['prefill_attention']} prefill, "
          f"graphs {'on' if e['decode_cuda_graphs'] else 'OFF'}, "
          f"pool {e['num_blocks']*e['block_size']} tokens, warm-up {e['warmup_s']}s")

    points = pd.DataFrame([p for p in r["points"] if p.get("ok")])
    if not len(points):
        print("no valid points"); continue
    table = pd.DataFrame({
        "batch": points.batch, "ctx": points.context,
        "ms/step": points.step_ms.round(3),
        "tok/s": points.tokens_per_s_aggregate.round(1),
        "MB/step (ideal)": (points.bytes_ideal / 1e6).round(1),
        "MB/step (impl)": (points.bytes_implemented / 1e6).round(1),
        "KV share %": (100 * points.kv_bytes_ideal / points.bytes_ideal).round(1),
        "GB/s (ideal)": points.achieved_gb_per_s_ideal.round(1),
        "GB/s (impl)": points.achieved_gb_per_s_implemented.round(1),
        "% of probe (ideal)": points.pct_of_probe_ideal.round(1),
        "% of probe (impl)": points.pct_of_probe_implemented.round(1),
        "floor ms (ideal)": points.floor_ms_ideal.round(3),
        "× floor (ideal)": points.multiple_of_floor_ideal.round(2),
        "windows clean": points.windows_clean.astype(str) + "/" + points.windows.astype(str),
    })
    display(table)
    table.to_csv(f"{WORKDIR}/bandwidth_{model}.csv", index=False)

if bw:
    fig, axes = plt.subplots(1, 2 * len(bw), figsize=(6.5 * 2 * len(bw), 4.2), squeeze=False)
    for i, (model, r) in enumerate(bw.items()):
        points = pd.DataFrame([p for p in r["points"] if p.get("ok")])
        if not len(points):
            continue
        rate = r["achieved_gb_per_s"]
        ax = axes[0][2 * i]
        for batch, df in points.groupby("batch"):
            ax.plot(df.context, df.achieved_gb_per_s_ideal, "o-", label=f"batch {batch}")
        ax.axhline(rate, color="k", ls="--", lw=1, label=f"gemv probe {rate:.0f} GB/s")
        ax.set_xscale("log", base=2); ax.set_xlabel("context tokens per row")
        ax.set_ylabel("achieved GB/s (ideal bytes)"); ax.set_title(f"{model}: decode bandwidth")
        ax.legend(fontsize=8)
        ax = axes[0][2 * i + 1]
        for batch, df in points.groupby("batch"):
            ax.plot(df.context, df.multiple_of_floor_ideal, "o-", label=f"batch {batch}")
        ax.axhline(1.0, color="k", ls="--", lw=1, label="the floor")
        ax.set_xscale("log", base=2); ax.set_xlabel("context tokens per row")
        ax.set_ylabel("step time / floor"); ax.set_title(f"{model}: multiple of the floor")
        ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(f"{WORKDIR}/bandwidth_part3.png", dpi=130); plt.show()

**How to read it.**

`% of probe` is the fraction of this card's measured attainable bandwidth that a decode step
reaches. A memory-bound kernel that is doing its job lands high here; what is left over is launch
overhead, the non-attention parts of the step, and any traffic the byte count does not model.

`× floor` is the same statement as a latency: 1.0× means the step costs exactly what moving its
bytes costs. It is *not* a target of 1.0 — the floor ignores kernel launch, sampling, scheduler
work and the logits write, so a small fixed cost per step keeps it above 1.0 and the gap shrinks
as the bytes grow.

The direction to read the table in is **across context**, not down it. As context grows, KV bytes
grow linearly while weight bytes stay fixed, so a step that is weight-dominated at 128 tokens
becomes KV-dominated at 2,048. Where the ratio flattens is where the engine is at its roofline and
**the only remaining lever is moving fewer bytes** — quantised weights, not a better attention
kernel.

`windows clean` reports how many of the timed windows passed every validity check. If it is 0 for a
point, that point's step time is not a decode measurement and should be discarded.

## 3. Summary block

The cell below prints one compact block (and writes `part3_summary.json`) with everything needed to
update the engineering report's memory and roofline sections. Paste it back.

In [ ]:
summary = {"gpu": None, "repo_commit": None, "memory": {}, "vllm_reference": {}, "bandwidth": {}}
summary["repo_commit"] = subprocess.run(
    ["git", "log", "-1", "--format=%h %ad %s", "--date=short"], cwd=REPO_DIR,
    capture_output=True, text=True).stdout.strip() or f"from {REPO_ZIP}"

for path in sorted(glob.glob(f"{WORKDIR}/results_mem/*.json")):
    r = json.load(open(path))
    if not r.get("ok"):
        continue
    summary["gpu"] = summary["gpu"] or r.get("gpu")
    model = r["model"].split("/")[-1]
    if r["variant"] == "vllm_reference":
        summary["vllm_reference"][model] = {
            "resident_mib": r["resident_mib"], "peak_mib": r["peak_mib"],
            "kv_capacity_matched": r["kv_capacity_matched"], "vllm": r["vllm_version"],
            "speed": {k: round(v, 2) for k, v in r["speed"].items()}}
        continue
    summary["memory"].setdefault(model, {})[r["variant"]] = {
        "resident_mib": r["resident_mib"], "peak_mib": r["peak_mib"],
        "attribution_mib": r["attribution_mib"],
        "graphs": r["warmup_summary"], "graph_geometry": r["graph_geometry"],
        "warmup_s": r["warmup_s"], "context_buckets": len(r["context_buckets_used"]),
        "speed": {k: (round(v, 2) if isinstance(v, float) else v) for k, v in r["speed"].items()},
    }

for path in sorted(glob.glob(f"{WORKDIR}/results_bw/*.json")):
    r = json.load(open(path))
    if not r.get("ok"):
        continue
    model = r["model"].split("/")[-1]
    summary["bandwidth"][model] = {
        "probes_gb_per_s": {p["name"]: round(p["gb_per_s"], 1) for p in r["probes"]},
        "gemv_gb_per_s": round(r["achieved_gb_per_s"], 1),
        "weight_bytes_read_per_step": r["weight_bytes"]["read_per_decode_step"],
        "geometry": r["geometry"], "engine": r["engine"],
        "points": [{k: (round(v, 4) if isinstance(v, float) else v)
                    for k, v in p.items() if k != "step_ms_all"} for p in r["points"]],
    }

with open(f"{WORKDIR}/part3_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"### part 3 summary — {summary['gpu']} — {summary['repo_commit']}\n")
for model, variants in summary["memory"].items():
    ref = summary["vllm_reference"].get(model, {}).get("resident_mib")
    variants = {v: variants[v] for v in VARIANTS if v in variants}
    print(f"{model}: " + ", ".join(
        f"{v}={d['resident_mib']} MiB ({d['graphs']['graphs']}/{d['graphs']['prefill_graphs']}"
        f"/{d['graphs']['fused_graphs']} graphs, {d['warmup_s']}s, "
        f"gather {d['graph_geometry']['largest_gather_mib_per_layer']} MiB)"
        for v, d in variants.items()))
    if ref:
        print(f"    vLLM reference {ref} MiB")
for model, d in summary["bandwidth"].items():
    print(f"{model}: gemv probe {d['gemv_gb_per_s']} GB/s, weights "
          f"{d['weight_bytes_read_per_step']/1e6:.1f} MB/step")
    for p in d["points"]:
        if p.get("ok"):
            print(f"    b{p['batch']:<3d} ctx{p['context']:<5d} {p['step_ms']:7.3f} ms  "
                  f"{p['achieved_gb_per_s_ideal']:6.1f} GB/s  "
                  f"{p['pct_of_probe_ideal']:5.1f}% of probe  "
                  f"{p['multiple_of_floor_ideal']:.2f}x floor  "
                  f"clean {p['windows_clean']}/{p['windows']}")

!cd {WORKDIR} && zip -qr /content/part3_results.zip results_mem results_bw logs *.csv *.png part3_summary.json 2>/dev/null; ls -la /content/part3_results.zip
print("\nDownload /content/part3_results.zip, and paste the block above back into the chat.")